# Data Leakage

## Learning Objectives
1. Reproduce the classic scaler-leakage bug and measure its inflated accuracy.
2. Quantify leakage using sklearn Pipeline vs manual preprocessing.
3. Demonstrate target-encoding leakage and how to prevent it.
4. Build a leakage audit tool that flags suspicious columns and temporal patterns.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Scaler Leakage Demo

In [ ]:
# The bug: fit StandardScaler on ALL data before the train/test split.
# The fix: fit scaler ONLY on training data.

X_leak, y_leak = make_classification(n_samples=500, n_features=20,
                                      n_informative=5, random_state=42)
split = 400

# BUG: scale the full dataset BEFORE splitting (test stats contaminate scaler)
scaler_bug = StandardScaler().fit(X_leak)
X_bug = scaler_bug.transform(X_leak)
clf_bug = LogisticRegression(max_iter=500)
clf_bug.fit(X_bug[:split], y_leak[:split])
acc_bug = clf_bug.score(X_bug[split:], y_leak[split:])

# FIX: fit scaler only on training portion
scaler_fix = StandardScaler().fit(X_leak[:split])
X_fix_train = scaler_fix.transform(X_leak[:split])
X_fix_test  = scaler_fix.transform(X_leak[split:])
clf_fix = LogisticRegression(max_iter=500)
clf_fix.fit(X_fix_train, y_leak[:split])
acc_fix = clf_fix.score(X_fix_test, y_leak[split:])

print(f"Leaky  accuracy: {acc_bug:.4f}  -- inflated (scaler saw test data)")
print(f"Correct accuracy: {acc_fix:.4f}  -- honest estimate")
print(f"Inflation: {acc_bug - acc_fix:+.4f}")
print("Even a small inflation can lead to wrong model-selection decisions.")


## Level 2: Pipeline vs Manual Preprocessing (Leakage Quantified)

In [ ]:
# Pipeline automatically fits preprocessors inside each CV fold.
# Manual preprocessing outside CV leaks validation statistics.

X_ppl, y_ppl = make_classification(n_samples=300, n_features=15,
                                    n_informative=6, random_state=7)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Leaky: scaler fitted on all data before CV
scaler_out = StandardScaler().fit(X_ppl)
X_scaled_all = scaler_out.transform(X_ppl)
scores_leaky = cross_val_score(LogisticRegression(max_iter=500),
                               X_scaled_all, y_ppl, cv=cv, scoring="accuracy")

# Correct: Pipeline ensures scaler is re-fitted per fold
pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])
try:
    scores_pipe = cross_val_score(pipe, X_ppl, y_ppl, cv=cv, scoring="accuracy")
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print("OOM -- reduce n_samples")
        torch.cuda.empty_cache()
    raise

print(f"Leaky (scaler outside CV): {scores_leaky.mean():.4f} +/- {scores_leaky.std():.4f}")
print(f"Correct (Pipeline):        {scores_pipe.mean():.4f} +/- {scores_pipe.std():.4f}")
print(f"Optimism bias:             {scores_leaky.mean() - scores_pipe.mean():+.4f}")
print("Rule: wrap ALL preprocessing steps in a Pipeline -- it is the only safe default.")


## Real-World Example 1: Target Encoding Leakage

In [ ]:
# Target encoding replaces a category with the mean target value.
# Global encoding leaks test-set labels into training features (BUG).

def target_encode_leaky(X_cat, y_tr, X_cat_test):
    # Global mean encoding -- uses test categories to compute means (BUG).
    all_cats = np.unique(np.concatenate([X_cat, X_cat_test]))
    means = {}
    for cat in all_cats:
        mask = X_cat == cat
        means[cat] = y_tr[mask].mean() if mask.any() else 0.5
    return np.array([means[c] for c in X_cat]), np.array([means[c] for c in X_cat_test])

def target_encode_correct(X_cat_tr, y_tr, X_cat_te):
    # Encoding computed only from training data (FIX).
    global_mean = y_tr.mean()
    means = {cat: y_tr[X_cat_tr == cat].mean() for cat in np.unique(X_cat_tr)}
    tr_enc = np.array([means[c] for c in X_cat_tr])
    te_enc = np.array([means.get(c, global_mean) for c in X_cat_te])
    return tr_enc, te_enc

# Synthetic data: 5 categories with correlated target
N_te = 300
cats = np.random.choice(["A","B","C","D","E"], N_te)
cat_effect = {"A": 0.2, "B": 0.4, "C": 0.6, "D": 0.8, "E": 0.9}
y_te_arr = (np.array([cat_effect[c] for c in cats]) + np.random.randn(N_te)*0.2 > 0.5).astype(int)
cats_tr, cats_test = cats[:240], cats[240:]
y_tr_arr, y_te_true = y_te_arr[:240], y_te_arr[240:]

for tag, fn in [("Leaky", target_encode_leaky), ("Correct", target_encode_correct)]:
    enc_tr, enc_te = fn(cats_tr, y_tr_arr, cats_test)
    clf = LogisticRegression()
    clf.fit(enc_tr.reshape(-1, 1), y_tr_arr)
    acc = clf.score(enc_te.reshape(-1, 1), y_te_true)
    print(f"{tag:10s} accuracy: {acc:.4f}")
print("Correct encoding uses only training-fold target means -- no leakage.")


## Real-World Example 2: Temporal Leakage in Time Series

In [ ]:
# Temporal leakage: using future-shifted features when predicting today's value.
# Common bug: forward-shifted target included as a feature.

np.random.seed(42)
T_ts = 400
sales = np.cumsum(np.random.randn(T_ts)) + 100

def make_leaky_features(sales, window=5):
    X, y = [], []
    for i in range(window, len(sales) - 1):
        feats = list(sales[i-window:i]) + [sales[i+1]]   # BUG: uses future value
        X.append(feats)
        y.append(sales[i])
    return np.array(X), np.array(y)

def make_correct_features(sales, window=5):
    X, y = [], []
    for i in range(window, len(sales)):
        X.append(sales[i-window:i].tolist())              # FIX: past-only
        y.append(sales[i])
    return np.array(X), np.array(y)

for tag, make_fn in [("Leaky", make_leaky_features), ("Correct", make_correct_features)]:
    X_all, y_all = make_fn(sales)
    sp = int(len(y_all) * 0.8)
    mdl = Ridge()
    mdl.fit(X_all[:sp], y_all[:sp])
    r2 = r2_score(y_all[sp:], mdl.predict(X_all[sp:]))
    print(f"{tag:10s} test R2: {r2:.4f}")

print("Leaky R2 near 1.0 -- the model is memorising tomorrow's value (future feature).")


## Real-World Example 3: Leakage Audit Tool

In [ ]:
# Automated leakage audit: flag features with suspiciously high target correlation.
# Use as a first-pass check before model training.

def audit_leakage(X, y, feature_names=None, threshold=0.9):
    # Flag features whose absolute Pearson correlation with y >= threshold.
    # Returns a list of (idx, name, correlation) tuples.
    if feature_names is None:
        feature_names = [f"feat_{i}" for i in range(X.shape[1])]
    flagged = []
    for i, name in enumerate(feature_names):
        corr, _ = pearsonr(X[:, i], y)
        if abs(corr) >= threshold:
            flagged.append((i, name, corr))
    return sorted(flagged, key=lambda t: abs(t[2]), reverse=True)

np.random.seed(99)
N_audit = 500
X_clean = np.random.randn(N_audit, 8)
y_audit = (X_clean[:, 0] + np.random.randn(N_audit) * 0.5 > 0).astype(float)

# Inject two suspicious features: noisy copies of the target
X_leaked = np.column_stack([
    X_clean,
    y_audit + np.random.randn(N_audit) * 0.05,   # feature 8: near-perfect proxy
    y_audit + np.random.randn(N_audit) * 0.2,    # feature 9: noisy proxy
])
feat_names = [f"feat_{i}" for i in range(8)] + ["proxy_exact", "proxy_noisy"]

flagged = audit_leakage(X_leaked, y_audit, feat_names, threshold=0.8)
print(f"Flagged {len(flagged)} suspicious features:")
for idx, name, corr in flagged:
    print(f"  Feature '{name}' (idx={idx}): corr={corr:.4f}")

# Verify clean data gets no flags
clean_flagged = audit_leakage(X_clean, y_audit, threshold=0.8)
print(f"Clean data flagged: {len(clean_flagged)}  (expected 0)")
print("Audit tool correctly identifies proxy features that would cause leakage.")
